In [ ]:
# walker / pedestrian cluster stats (training set, from distribution_analysis.ipynb)
#   walker_adult: 43 IDENTICAL capsules -> sigma == 0 -> RELATIVE-DIFFERENCE method
#   walker_child:  8 capsules with spread -> sigma  > 0 -> Z-SCORE method
# Walkers use UNION-of-clusters: a candidate is a visual shift if it fits EITHER human
# mold, and a geometric shift only if it fits NEITHER (Executive Summary 2B.2-3).
ADULT_MU = {"L": 0.375400, "W": 0.375400, "H": 1.860000}                 # sigma == 0
CHILD    = {"L": (0.453275, 0.064487),                                   # (mu, sigma)
            "W": (0.453275, 0.064487),
            "H": (1.175000, 0.103510)}

Z_VISUAL            = 2.0    # walker_child Z-score visual cutoff  (Z <= 2 -> visual)
Z_GEOMETRIC         = 3.0    # walker_child Z-score geometric cutoff (Z > 3 -> geometric)
DELTA_VISUAL_WALKER = 0.20   # walker_adult relative-diff cutoff (binary: <=20% visual else geometric)
HEIGHT_SATURATION   = 2.0    # carried from the vehicle/notebook logic (no-op here: both mu_H < 2.0m)


def carla_bbox_from_capsule(radius_cm: float, half_height_cm: float):
    """UE capsule collision -> CARLA bounding-box (full extents), in metres.

    Read 'Capsule Radius' and 'Capsule Half Height' (cm) straight off the UE Shape panel.
    CARLA reports the bbox as full extents, and the walker capsule maps as:
        L = W = 2 * radius        (capsule diameter)
        H     = 2 * half_height   (full capsule height)
    then cm -> m (/100).  Verified on walker.pedestrian.astronaut:
        radius=21.3, half_height=93  ->  L=W=0.426, H=1.86  (== CARLA's reported bbox).
    """
    L = W = 2.0 * radius_cm / 100.0
    H = 2.0 * half_height_cm / 100.0
    return {"L": L, "W": W, "H": H}


def _eval_adult(bbox):
    """walker_adult is zero-variance -> relative difference d = |x - mu| / mu, 20% cutoff."""
    rows, deltas = [], {}
    for dim in ("L", "W", "H"):
        mu = ADULT_MU[dim]
        d = abs(bbox[dim] - mu) / mu
        deltas[dim] = d
        lo, hi = mu * (1 - DELTA_VISUAL_WALKER), mu * (1 + DELTA_VISUAL_WALKER)
        if d <= DELTA_VISUAL_WALKER:
            status = f"OK, {(DELTA_VISUAL_WALKER - d) * 100:4.1f}pp headroom"
        else:
            status = f"OVER by {(d - DELTA_VISUAL_WALKER) * 100:4.1f}pp"
        rows.append(f"  d_{dim} = {d * 100:5.1f}%  [{status}]   visual range: {lo:.4f} .. {hi:.4f}")
    dmax = max(deltas, key=deltas.get)
    shift = "level_1_visual" if deltas[dmax] <= DELTA_VISUAL_WALKER else "level_2_geometric"
    return rows, dmax, deltas[dmax], shift


def _eval_child(bbox):
    """walker_child has spread -> Z-score = |x - mu| / sigma, 2-sigma / 3-sigma bands."""
    rows, zs = [], {}
    for dim in ("L", "W", "H"):
        mu, sigma = CHILD[dim]
        if dim == "H" and mu > HEIGHT_SATURATION and bbox[dim] > HEIGHT_SATURATION:
            z = 0.0
        else:
            z = abs(bbox[dim] - mu) / sigma
        zs[dim] = z
        lo, hi = mu - Z_VISUAL * sigma, mu + Z_VISUAL * sigma
        if z <= Z_VISUAL:
            status = f"OK, headroom {Z_VISUAL - z:4.2f}sigma"
        elif z > Z_GEOMETRIC:
            status = f"OVER by {z - Z_VISUAL:4.2f}sigma"
        else:
            status = f"ambiguous (+{z - Z_VISUAL:4.2f}sigma)"
        rows.append(f"  Z_{dim} = {z:5.2f}  [{status}]   visual range: {lo:.4f} .. {hi:.4f}")
    zmaxd = max(zs, key=zs.get)
    zmax = zs[zmaxd]
    if zmax <= Z_VISUAL:
        shift = "level_1_visual"
    elif zmax > Z_GEOMETRIC:
        shift = "level_2_geometric"
    else:
        shift = "ambiguous"
    return rows, zmaxd, zmax, shift


_LABEL = {"level_1_visual": "VISUAL (Level 1)",
          "level_2_geometric": "GEOMETRIC (Level 2)",
          "ambiguous": "AMBIGUOUS"}


def check_walker(name: str, radius: float, half_height: float) -> None:
    """Classify a candidate pedestrian asset from its UE capsule (radius, half_height in cm).

    radius      = 'Capsule Radius'      from the UE Shape panel (cm)
    half_height = 'Capsule Half Height' from the UE Shape panel (cm)
    """
    bbox = carla_bbox_from_capsule(radius, half_height)
    a_rows, a_dim, a_val, a_shift = _eval_adult(bbox)
    c_rows, c_dim, c_val, c_shift = _eval_child(bbox)

    if a_shift == "level_1_visual" or c_shift == "level_1_visual":
        combined = "level_1_visual"
    elif a_shift == "level_2_geometric" and c_shift == "level_2_geometric":
        combined = "level_2_geometric"
    else:
        combined = "ambiguous"

    print(f"\n{'=' * 70}")
    print(f"{name}  (capsule: radius={radius:g}cm, half_height={half_height:g}cm)")
    print(f"Derived CARLA bbox: L={bbox['L']:.4f}, W={bbox['W']:.4f}, H={bbox['H']:.4f}  (m)")
    if bbox["H"] > 2.5:
        print("  ! note: derived height > 2.5m - did you pass the FULL height instead of half-height?")
    print(f"{'-' * 70}")
    print(f"vs walker_adult  [relative-difference, sigma=0]   "
          f"mu: L={ADULT_MU['L']:.4f} W={ADULT_MU['W']:.4f} H={ADULT_MU['H']:.4f}")
    for r in a_rows:
        print(r)
    print(f"  -> d_max = {a_val * 100:.1f}% ({a_dim})  ->  {_LABEL[a_shift]}")
    print(f"{'-' * 70}")
    print(f"vs walker_child  [Z-score]   "
          f"mu: L={CHILD['L'][0]:.4f} W={CHILD['W'][0]:.4f} H={CHILD['H'][0]:.4f}  "
          f"sigma: L={CHILD['L'][1]:.4f} W={CHILD['W'][1]:.4f} H={CHILD['H'][1]:.4f}")
    for r in c_rows:
        print(r)
    print(f"  -> Z_max = {c_val:.2f} ({c_dim})  ->  {_LABEL[c_shift]}")
    print(f"{'-' * 70}")
    print(f"  UNION VERDICT:  {_LABEL[combined]}")
    if combined == "level_1_visual":
        which = "walker_adult" if a_shift == "level_1_visual" else "walker_child"
        print(f"    fits {which} -> visual.  (OR-rule: visual if it fits EITHER human mold.)")
    elif combined == "level_2_geometric":
        print(f"    fits NEITHER mold (adult & child both geometric) -> AND-rule -> geometric.")
    else:
        print(f"    neither a clean visual fit nor geometric on both -> ambiguous; review manually.")
    print(f"{'=' * 70}")


## Pedestrian / walker dimension checker

Pre-design eligibility check for a **walker.pedestrian** asset, driven by its Unreal Engine
**capsule collision** (the same `radius` + `half_height` you set in the Shape panel), so you can
validate the shift level *before* finalizing the asset.

### Inputs (read straight off the UE Shape panel, in cm)
- `radius`      = **Capsule Radius**
- `half_height` = **Capsule Half Height**

### How CARLA turns the capsule into a bounding box
CARLA reports the walker bbox as full extents, so the capsule maps as:

| CARLA bbox | from capsule | cm → m |
|---|---|---|
| `L` = `W` | `2 × radius` (diameter) | `/ 100` |
| `H` | `2 × half_height` (full capsule height) | `/ 100` |

Sanity check — `walker.pedestrian.astronaut` (radius=21.3, half_height=93) → `L=W=0.426, H=1.86`,
exactly the dimensions CARLA reports for it.

### Classification — union of two human molds
Unlike vehicles (one `vehicle_car` cluster), walkers are scored against **both** training clusters:

- **`walker_adult`** (43 identical capsules, σ = 0) — uses **relative difference** `Δ = |x−μ|/μ`,
  visual if `Δ ≤ 20%`, else geometric.
- **`walker_child`** (8 capsules with spread) — uses **Z-score** `|x−μ|/σ`,
  visual if `Z ≤ 2`, geometric if `Z > 3`, otherwise ambiguous.

The two are combined with the **union rule**:
- **Visual (Level 1)** if it fits *either* mold (OR).
- **Geometric (Level 2)** only if it fits *neither* mold (AND).
- **Ambiguous** otherwise — review manually.

Edit the call in the last cell to your candidate's capsule values and **re-run it**.


In [ ]:
# Edit the call below to your candidate's capsule values, then re-run the cell.
# radius / half_height are the 'Capsule Radius' / 'Capsule Half Height' fields (cm) in UE.
#
# Reference / in-distribution walkers (round-trip checks against CARLA's reported bbox):
# check_walker("astronaut",     radius=21.3, half_height=93)    # -> VISUAL (Level 1)
# check_walker("firefighter",   radius=19.0, half_height=93)    # -> VISUAL (Level 1)
# check_walker("soldier",       radius=21.8, half_height=93)    # -> VISUAL (Level 1)
#
# Geometric-shift examples (fit neither the adult nor the child human mold):
# check_walker("deliveryrobot", radius=43.2, half_height=65)    # -> GEOMETRIC (Level 2)
# check_walker("wheelchair",    radius=44.0, half_height=70)    # -> GEOMETRIC (Level 2)

# check_walker("astronaut", radius=21.3, half_height=93)
# check_walker("caneman", radius=21, half_height=84.5)
# check_walker("coneman", radius=40.327, half_height=40.327)
check_walker("cow", radius=40, half_height=75)
